# qradiomics: IBSI-1 digital phantom validation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/choilab-jefferson/qradiomics/blob/main/examples/ibsi1_phantom_colab.ipynb)

The Image Biomarker Standardisation Initiative (IBSI) defines a small **digital
phantom** (chapter 2 of the IBSI-1 reference manual) with consensus feature values.
This notebook runs the native `rtools` engine's IBSI-1 profile on the phantom and
shows, per feature, whether it agrees with the reference.

No data download from an imaging archive and no credentials: the phantom and the
reference values are one small JSON file from the qradiomics repository.

## 0. Install

In [ ]:
!pip install -qU qradiomics
!qr --version

## 1. Fetch the phantom and reference values

The JSON holds the phantom (5 x 4 x 4 voxels in x, y, z, integer grey levels, and
its mask, 2 mm voxels) and the IBSI consensus values with tolerances for four
feature families: morphology, statistics, GLCM and GLRLM.

In [ ]:
import json
import urllib.request

import qradiomics

REPO_RAW = "https://raw.githubusercontent.com/choilab-jefferson/qradiomics"
NAME = "tests/fixtures/ibsi1_digital_phantom.json"
for ref in (f"v{qradiomics.__version__}", "main"):  # prefer the tag matching the installed version
    try:
        phantom = json.load(urllib.request.urlopen(f"{REPO_RAW}/{ref}/{NAME}"))
        print("phantom from", ref)
        break
    except Exception as exc:  # noqa: BLE001
        print(f"{ref}: {exc}")

print(phantom["source"])
print("spacing (x, y, z) mm:", phantom["spacing_xyz"], "| reference features:", len(phantom["reference"]))

## 2. Extract with the IBSI-1 profile

Configuration as in the IBSI benchmark: no interpolation, no re-segmentation, and
discretisation with a fixed bin size of 1 starting at the ROI minimum (the phantom's
integer grey levels).

In [ ]:
import numpy as np
from qradiomics.feature.rtools.ibsi import compare_reference, extract_ibsi_arrays

values = extract_ibsi_arrays(
    np.array(phantom["image_zyx"]), np.array(phantom["mask_zyx"]),
    phantom["spacing_xyz"], bin_width=1.0,
)
print(f"{len(values)} features computed")

## 3. Pass / fail against the IBSI reference

A feature passes when it is within the reference tolerance, or when both values agree
to 3 significant figures (the precision the IBSI tables are published at).

In [ ]:
import pandas as pd

rows = compare_reference(values, phantom["reference"])
df = pd.DataFrame(rows, columns=["feature", "value", "reference", "tolerance", "pass"])
df["family"] = df.feature.str.split("_").str[0]

FAMILY = {"morph": "morphology", "stat": "statistics", "cm": "GLCM", "rlm": "GLRLM"}
summary = (df.groupby("family")["pass"].agg(passed="sum", total="count")
             .rename(index=lambda f: f"{f} ({FAMILY.get(f, f)})"))
summary["passed"] = summary["passed"].astype(int)
print(f"PASS {int(df['pass'].sum())} / {len(df)} reference features")
summary

In [ ]:
# The full table: one row per reference feature (failures sort first).
df.sort_values("pass").reset_index(drop=True)[["feature", "value", "reference", "tolerance", "pass"]]

In [ ]:
failed = df[~df["pass"]]
assert failed.empty, f"{len(failed)} feature(s) outside tolerance:\n{failed}"
print("All implemented IBSI-1 reference features agree with the consensus values.")

## Notes

* The IBSI profile is the opt-in `rtools_ibsi=True` mode of the native engine.
  The default `rtools` engine instead reproduces the legacy radiomics-tools
  `FeatureExtraction` binary, which follows its own conventions.
* The same check runs in the qradiomics test suite (`tests/test_rtools_ibsi.py`).
* Reference: Zwanenburg A *et al.*, The Image Biomarker Standardization Initiative,
  Radiology 2020;295:328-338.

## Optional: drive this with an agent

Everything above runs by hand. If you would rather have an agent (Antigravity `agy`,
Claude Code, ...) run the same `qr` commands for you inside this Colab runtime,
install the small `nexus-lite` helper and prepare the environment. This section is
optional; skip it if you only want the notebook.

Agent skills and the full guide live in
[choilab-jefferson/nexus.lite](https://github.com/choilab-jefferson/nexus.lite)
(see `skills/nexus-colab`). The qradiomics skill that teaches an agent the `qr` CLI
ships with this repository under `.claude/skills/qradiomics/` and
`.agents/skills/qradiomics/`.

In [ ]:
!pip install -qU nexus-lite
from nexus_lite import colab
colab.ensure_env()